# 🧩 Notebook 3 · Online Stock Brokerage — Patterns & Matching Engine

## 🛠️ Setup

```bash
cd 07-object-oriented-design/online-stock-brokerage
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In this notebook we upgrade the toy brokerage with three classic OO patterns and a more realistic matching engine.

We'll go **bad → best** for each concept:

1. **Order book** — from a flat list to **price-time priority** matching.
2. **Observer** — push live P&L updates without coupling.
3. **Decorator** — add risk checks (max position size, cash guard) without touching `Exchange`.

These three ideas cover most of the OO design feedback you'd receive in an interview for this problem.


## 1️⃣ Shared domain (same model as Notebook 2)

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from itertools import count
import heapq

@dataclass
class Stock:
    symbol: str
    price: float

class Side(Enum):
    BUY = "buy"; SELL = "sell"

class OrderStatus(Enum):
    PENDING="pending"; FILLED="filled"; PARTIAL="partial"; CANCELLED="cancelled"

@dataclass
class Position:
    symbol: str; qty: int = 0; avg_price: float = 0.0

@dataclass
class Portfolio:
    positions: dict[str, Position] = field(default_factory=dict)
    def apply(self, symbol, qty_delta, price):
        pos = self.positions.setdefault(symbol, Position(symbol))
        if qty_delta > 0:
            new_qty = pos.qty + qty_delta
            pos.avg_price = (pos.avg_price*pos.qty + price*qty_delta)/new_qty
            pos.qty = new_qty
        else:
            pos.qty += qty_delta
        if pos.qty == 0: self.positions.pop(symbol, None)

@dataclass
class Account:
    id: int; cash: float
    portfolio: Portfolio = field(default_factory=Portfolio)


## 2️⃣ Order book — attempt 0: a flat list (bad 🔴)

The Notebook-2 `Exchange` stored every order in `self.book: list[Order]` and scanned it on every price tick. That's `O(n)` per tick and says nothing about **who gets filled first**.

In real markets, the rule is **price-time priority**:

- Best price wins (highest bid, lowest ask).
- Among orders at the same price, **earliest submitted** wins.

A flat list has neither property.


## 3️⃣ Order book — best 🟢: two price-ordered heaps

We keep two heaps (priority queues):

- **Bids** (buyers): max-heap on price — highest willing-to-pay wins.
- **Asks** (sellers): min-heap on price — lowest asking seller wins.

A trade happens whenever `best_bid.price >= best_ask.price`. We break price ties by **submission time** (a monotonic counter), giving us price-time priority for free.


In [ ]:
_oids = count(1)
_seq  = count(1)          # global submission sequence (for time priority)

@dataclass(order=True)
class BookEntry:
    # Heap sort key - put sort-relevant fields first, then tiebreakers.
    sort_key: tuple
    order_id: int = field(compare=False)
    account:  Account = field(compare=False)
    symbol:   str = field(compare=False)
    side:     Side = field(compare=False)
    price:    float = field(compare=False)
    qty:      int = field(compare=False)

@dataclass
class Trade:
    buy_id: int; sell_id: int
    symbol: str; qty: int; price: float
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

class OrderBook:
    """Per-symbol limit order book with price-time priority."""
    def __init__(self, symbol: str):
        self.symbol = symbol
        self.bids: list[BookEntry] = []   # max-heap via negated price
        self.asks: list[BookEntry] = []   # min-heap
        self.trades: list[Trade] = []

    def submit(self, account: Account, side: Side, qty: int, price: float) -> int:
        oid = next(_oids)
        t = next(_seq)                    # earlier submissions get smaller t
        if side == Side.BUY:
            # Highest price first; at same price, earliest time first.
            heapq.heappush(self.bids, BookEntry((-price, t), oid, account, self.symbol, side, price, qty))
        else:
            heapq.heappush(self.asks, BookEntry(( price, t), oid, account, self.symbol, side, price, qty))
        self._match()
        return oid

    def _match(self) -> None:
        while self.bids and self.asks and self.bids[0].price >= self.asks[0].price:
            bid = self.bids[0]
            ask = self.asks[0]
            # Trade price = resting order's price (the one that was in the book first).
            trade_price = ask.price if bid.sort_key[1] > ask.sort_key[1] else bid.price
            traded_qty = min(bid.qty, ask.qty)

            # Settle accounts
            cost = trade_price * traded_qty
            bid.account.cash -= cost
            bid.account.portfolio.apply(self.symbol, +traded_qty, trade_price)
            ask.account.cash += cost
            ask.account.portfolio.apply(self.symbol, -traded_qty, trade_price)

            self.trades.append(Trade(bid.order_id, ask.order_id, self.symbol, traded_qty, trade_price))

            # Reduce / remove filled sides
            bid.qty -= traded_qty; ask.qty -= traded_qty
            if bid.qty == 0: heapq.heappop(self.bids)
            if ask.qty == 0: heapq.heappop(self.asks)

    def best_bid(self): return self.bids[0].price if self.bids else None
    def best_ask(self): return self.asks[0].price if self.asks else None


### Demo — price-time priority in action

In [ ]:
aapl_book = OrderBook("AAPL")
alice = Account(1, cash=100_000)
bob   = Account(2, cash=100_000)
carol = Account(3, cash=0,     portfolio=Portfolio({"AAPL": Position("AAPL", qty=50, avg_price=150)}))
bob.portfolio.positions["AAPL"] = Position("AAPL", qty=50, avg_price=150)

# Two sellers at the same price. Carol submits first -> she should get filled first.
aapl_book.submit(carol, Side.SELL, 20, price=180)
aapl_book.submit(bob,   Side.SELL, 20, price=180)   # sells only if carol is done

# A buyer sweeps 25 shares at 180.
aapl_book.submit(alice, Side.BUY, 25, price=180)

for t in aapl_book.trades:
    print(t)
print("Carol qty left:", carol.portfolio.positions.get("AAPL"))
print("Bob   qty left:", bob.portfolio.positions.get("AAPL"))
print("Alice owns    :", alice.portfolio.positions.get("AAPL"))
print("best_bid:", aapl_book.best_bid(), "best_ask:", aapl_book.best_ask())


Notice:

- Carol sold 20 (her full order) because she was first in time.
- Bob then sold 5 — the remainder of Alice's buy demand at price 180.
- Both trades printed at price **180** regardless of which side was aggressive.
- The book still has Bob's remaining 15 shares resting at the ask.


## 4️⃣ Observer — live quotes without coupling (bad → best)

**Bad 🔴:** the `OrderBook` directly calls a `print_quote()` and a `log_trade_to_db()` function. Now the book has to know about UIs and databases — every new consumer means editing the book.

**Best 🟢:** the book just publishes events. Anyone who cares **subscribes**. This is the **Observer pattern**.


In [ ]:
class TradeObserver(ABC):
    @abstractmethod
    def on_trade(self, trade: Trade) -> None: ...

class QuotePrinter(TradeObserver):
    """Prints a one-line ticker tape."""
    def on_trade(self, trade):
        print(f"  📈 {trade.symbol} {trade.qty}@{trade.price}")

class MarkToMarket(TradeObserver):
    """Prints account's mark-to-market value on every trade of a symbol."""
    def __init__(self, account: Account, symbol: str):
        self.account = account; self.symbol = symbol
    def on_trade(self, trade):
        if trade.symbol != self.symbol: return
        pos = self.account.portfolio.positions.get(self.symbol)
        mark = pos.qty * trade.price if pos else 0
        print(f"  💰 acct#{self.account.id}: {self.symbol} m2m={mark:.2f}  cash={self.account.cash:.2f}")

class ObservableOrderBook(OrderBook):
    def __init__(self, symbol):
        super().__init__(symbol)
        self._observers: list[TradeObserver] = []
    def subscribe(self, obs: TradeObserver):   self._observers.append(obs)
    def _match(self):
        before = len(self.trades)
        super()._match()
        for t in self.trades[before:]:
            for obs in self._observers:
                obs.on_trade(t)

# Demo: plug in observers at runtime - the book itself didn't change.
book = ObservableOrderBook("AAPL")
alice = Account(1, cash=100_000)
bob   = Account(2, cash=0, portfolio=Portfolio({"AAPL": Position("AAPL", qty=30, avg_price=150)}))

book.subscribe(QuotePrinter())
book.subscribe(MarkToMarket(alice, "AAPL"))

book.submit(bob,   Side.SELL, 10, price=182)
book.submit(alice, Side.BUY,  10, price=182)


### Why this is better 🟢

- The `OrderBook` **doesn't import** `QuotePrinter` or `MarkToMarket`.
- You can add a `SlackAlerter`, `CsvRecorder`, `RiskMonitor` without touching the book.
- Tests can subscribe a fake observer and assert events were published.


## 5️⃣ Decorator — risk checks without touching `OrderBook`

**Bad 🔴:** add `if qty > max_position: reject` directly inside `submit`. Every new rule (margin, PDT, KYC) grows the method.

**Best 🟢:** wrap the book in a **decorator** (same interface, extra behaviour). Stack as many rules as you like.


In [ ]:
class Broker(ABC):
    """Minimal interface that both the real book and decorators expose."""
    @abstractmethod
    def submit(self, account: Account, side: Side, qty: int, price: float) -> int: ...

class BookBroker(Broker):
    def __init__(self, book: OrderBook): self.book = book
    def submit(self, account, side, qty, price): return self.book.submit(account, side, qty, price)

class MaxPositionGuard(Broker):
    """Reject buys that would push a user past `limit` shares of `symbol`."""
    def __init__(self, inner: Broker, symbol: str, limit: int):
        self.inner, self.symbol, self.limit = inner, symbol, limit
    def submit(self, account, side, qty, price):
        if side == Side.BUY:
            have = account.portfolio.positions.get(self.symbol)
            current = have.qty if have else 0
            if current + qty > self.limit:
                raise PermissionError(f"position cap exceeded: {current}+{qty} > {self.limit}")
        return self.inner.submit(account, side, qty, price)

class SufficientCashGuard(Broker):
    """Reject buys the user can't pay for (no margin in this toy)."""
    def __init__(self, inner: Broker): self.inner = inner
    def submit(self, account, side, qty, price):
        if side == Side.BUY and account.cash < qty*price:
            raise PermissionError(f"insufficient cash: need {qty*price}, have {account.cash}")
        return self.inner.submit(account, side, qty, price)

# Compose rules like Russian dolls:
raw_book = OrderBook("AAPL")
broker: Broker = BookBroker(raw_book)
broker = SufficientCashGuard(broker)
broker = MaxPositionGuard(broker, "AAPL", limit=100)

alice = Account(1, cash=5_000)
bob   = Account(2, cash=0, portfolio=Portfolio({"AAPL": Position("AAPL", qty=200, avg_price=150)}))

# Seller - always allowed here (we only guard buys).
broker.submit(bob, Side.SELL, 50, price=180)

# 1) Alice tries to buy too many -> MaxPositionGuard blocks.
try:
    broker.submit(alice, Side.BUY, 120, price=180)
except PermissionError as e:
    print("blocked by position cap:", e)

# 2) Alice tries to buy 50 but only has $5,000 (needs $9,000) -> cash guard blocks.
try:
    broker.submit(alice, Side.BUY, 50, price=180)
except PermissionError as e:
    print("blocked by cash:", e)

# 3) Alice tops up cash and buys within limits -> goes through.
alice.cash = 20_000
oid = broker.submit(alice, Side.BUY, 50, price=180)
print("filled order id:", oid, "cash left:", alice.cash, "owns:", alice.portfolio.positions)


### Why the decorator wins 🟢

- Each rule is a tiny class that does **one thing** (SRP).
- Rules **compose** in any order — you can enable/disable them per user tier.
- `OrderBook.submit` stays clean and focused on matching.
- Easy to unit-test each guard in isolation with a stub inner broker.


## 6️⃣ Where to go next

You now have the bones of a trading system:

- **Order polymorphism** (Notebook 2) → clean fill rules.
- **Order book** with price-time priority → fair matching.
- **Observer** → plug in quotes, P&L, alerts, analytics.
- **Decorator** → layered risk/compliance checks.

Ideas to extend:

- Add `cancel(order_id)` to the heap-based book (mark-cancelled + lazy removal when popped).
- Support **market orders** in the new book by submitting at `price=+∞` (buys) / `-∞` (sells).
- Add an **audit log** observer that writes every trade to a CSV — great for reproducible tests.
- Introduce an `Account`-level **margin** model and a new decorator that enforces it.

All of these are additive — you won't have to edit the matching engine again. That's the whole point. 🎯
